# QuantumShield

## Noise-Resilient Quantum Classification & Error Mitigation

**Qiskit Fall Fest 2026 — Phase 1**

**Problem Statement:** PS-Q3

QuantumShield studies how synthetic device noise and processor
connectivity affect a 4-qubit Variational Quantum Classifier (VQC).

The project compares Processor A and Processor B under identical
noise conditions and evaluates readout error mitigation.


## 1. Experimental Configuration

- Dataset: synthetic binary classification
- Samples: 200
- Features: 4
- Train/test split: 80/20
- Quantum model: 4-qubit VQC
- Encoding: angle encoding
- Optimizer: COBYLA
- Transpiler optimization level: 1
- Transpiler seed: 42
- Shots: 4096

### Synthetic noise

- 1-qubit depolarizing error: 0.001
- CX depolarizing error: 0.005
- Readout error: 0.01


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.problem import prepare_q3_dataset
from src.vqc import build_vqc

ROOT = Path.cwd()
TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"

print("QuantumShield environment ready.")


## 2. Dataset

The classifier uses a deterministic 200-sample,
4-feature synthetic binary classification dataset.

The features are scaled to `[0, π]` for quantum angle encoding.


In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_q3_dataset()

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Features/sample :", X_train.shape[1])

print()
print("Training class counts:")
print(pd.Series(y_train).value_counts().sort_index())

print()
print("Testing class counts:")
print(pd.Series(y_test).value_counts().sort_index())


## 3. Variational Quantum Classifier

The logical classifier contains:

- 4 qubits
- 4 data-encoding parameters
- 8 trainable parameters
- RY angle encoding
- CX entanglement


In [ ]:
circuit, x_params, theta_params = build_vqc()

print(circuit.draw(output="text"))

print()
print("Logical depth:", circuit.depth())
print("Gate counts:", circuit.count_ops())


## 4. Ideal VQC Training

The VQC was trained once on an ideal statevector simulator.

The trained parameters were then frozen and reused for all
processor/noise experiments. This avoids retraining separately
for Processor A and Processor B.


In [ ]:
with open(TABLES / "ideal_vqc_summary.json", "r") as f:
    ideal_summary = json.load(f)

pd.DataFrame([ideal_summary])


### Training convergence


In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            FIGURES / "training_convergence.png"
        )
    )
)


## 5. Processor A vs Processor B — Classifier Routing

The same trained classifier is mapped to both processor
architectures.

For the classifier's simple chain-like interaction graph,
both processors can find an efficient physical mapping.


In [ ]:
routing = pd.read_csv(
    TABLES / "routing_benchmark.csv"
)

routing


## 6. Synthetic Noise Benchmark

The frozen VQC is evaluated with 4096 shots under the same
synthetic noise rates on Processor A and Processor B.

A single seed should not be interpreted in isolation because
finite-shot sampling can move probabilities near the 0.5
classification boundary.


In [ ]:
noisy = pd.read_csv(
    TABLES / "noisy_benchmark.csv"
)

noisy


## 7. Readout Error Mitigation

A symmetric readout-error correction is applied to the measured
class-1 probability.

Hard-label accuracy may remain unchanged because symmetric
readout correction preserves the 0.5 classification boundary.

Therefore mitigation is additionally evaluated using probability
error relative to the ideal statevector output.


In [ ]:
mitigation = pd.read_csv(
    TABLES / "mitigation_quality.csv"
)

mitigation


In [ ]:
display(
    Image(
        filename=str(
            FIGURES / "mitigation_mae.png"
        )
    )
)


## 8. Repeated-Seed Robustness

Five simulator seeds are used to test whether the mitigation
conclusion persists across finite-shot realizations.


In [ ]:
repeated = pd.read_csv(
    TABLES / "repeated_seed_summary.csv"
)

repeated


In [ ]:
display(
    Image(
        filename=str(
            FIGURES / "repeated_seed_accuracy.png"
        )
    )
)


## 9. Connectivity Stress Test

The original VQC uses a simple interaction chain, which maps
efficiently to both processors.

To isolate architectural effects, a separate 4-qubit cyclic
interaction workload is used as a routing stress test.

This stress circuit is **not a replacement for the trained
classifier**. It is used only for processor-connectivity analysis.


In [ ]:
architecture = pd.read_csv(
    TABLES / "architecture_stress.csv"
)

architecture


In [ ]:
display(
    Image(
        filename=str(
            FIGURES / "architecture_comparison.png"
        )
    )
)


## 10. Key Findings

### Classification

The trained 4-qubit VQC achieved **77.5% ideal test accuracy**.

Across five 4096-shot noisy simulations, mean classification
accuracy was approximately **79.0%**, with finite-shot variation.

The noisy accuracy should not be interpreted as evidence that
noise improves the classifier; the 40-sample test set contains
predictions near the 0.5 decision boundary.

### Error mitigation

Readout mitigation did not alter hard-label accuracy, but it
reduced the mean probability MAE relative to the ideal output
from approximately **0.009598 to 0.006276**, a relative reduction
of about **34.6%** across five simulator seeds.

### Processor architecture

The original chain-like classifier mapped equally efficiently
to both processors.

Under the cyclic connectivity stress test:

- Processor A required 7 CX gates and depth 11.
- Processor B required 4 CX gates and depth 8.

Thus Processor B eliminated the observed CX routing overhead for
this cyclic workload.

### Conclusion

QuantumShield demonstrates that error mitigation and processor
connectivity should be evaluated using more than classification
accuracy alone.

Readout mitigation improved probability-level agreement with the
ideal classifier, while the architecture experiment showed that
processor-connectivity advantages depend on the interaction
structure of the quantum circuit.
